# Convert the dense model and verify anchor equivalence

**Feature:** NB-002  
**Goal:** convert selected Qwen3 MLPs to forced-anchor wrappers and prove exact hidden-state, logit, and anchor-state equivalence.

This notebook is intentionally an offline CPU smoke test. It uses a deterministic, randomly initialized tiny Qwen3 model and does not download weights.

## Resolve configuration

The model, input, converted-layer selector, and numerical tolerances are declared once and reused by every validation cell.

In [ ]:
import torch
from transformers import Qwen3Config, Qwen3ForCausalLM

from anchored_moe import AnchoredMoEConfig, RoutedMLP
from anchored_moe.backends import Qwen3Backend
from anchored_moe.evaluation import assert_tensors_equivalent, fingerprint_module
from anchored_moe.utils import sha256_json

EXPERIMENT = {
    "feature_id": "NB-002",
    "seed": 7,
    "base_model_id": "local-random-qwen3-smoke",
    "base_model_revision": "synthetic-seed-7",
    "converted_layers": "upper_half",
    "atol": 0.0,
    "rtol": 0.0,
    "input_ids": [[1, 7, 11, 3], [2, 5, 13, 9]],
}
EXPERIMENT

## Build and pin a deterministic anchor

The configuration hash and selected MLP fingerprints are captured before conversion.

In [ ]:
torch.manual_seed(EXPERIMENT["seed"])
base_config = Qwen3Config(
    vocab_size=64,
    hidden_size=16,
    intermediate_size=32,
    num_hidden_layers=4,
    num_attention_heads=4,
    num_key_value_heads=2,
    head_dim=4,
    max_position_embeddings=32,
)
model = Qwen3ForCausalLM(base_config).eval()
input_ids = torch.tensor(EXPERIMENT["input_ids"])
anchored_config = AnchoredMoEConfig(
    base_model_name_or_path=EXPERIMENT["base_model_id"],
    base_model_revision=EXPERIMENT["base_model_revision"],
    base_config_sha256=sha256_json(base_config.to_dict()),
    tokenizer_name_or_path=EXPERIMENT["base_model_id"],
    tokenizer_revision=EXPERIMENT["base_model_revision"],
    backend="qwen3",
    converted_layers=EXPERIMENT["converted_layers"],
)
backend = Qwen3Backend()
selected_layers = anchored_config.resolve_converted_layers(
    base_config.num_hidden_layers
)
original_mlps = {
    index: model.model.layers[index].mlp for index in selected_layers
}
anchor_fingerprints = {
    index: fingerprint_module(module) for index, module in original_mlps.items()
}

## Execute the dense reference and convert

Both executions use the same model object, input, evaluation mode, device, and dtype. The wrapper's forced-anchor path calls the original MLP directly.

In [ ]:
with torch.no_grad():
    dense_output = model(input_ids=input_ids, output_hidden_states=True)

conversion_report = backend.convert(model, anchored_config)

with torch.no_grad():
    converted_output = model(input_ids=input_ids, output_hidden_states=True)

assert conversion_report.converted_layers == selected_layers
trainable_parameters_after_conversion = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
assert trainable_parameters_after_conversion == 0

## Validate exact equivalence and immutability

CPU forced-anchor execution is required to be bitwise exact: both tolerances are zero, every hidden state is checked, and each moved anchor MLP must retain its original fingerprint and object identity.

In [ ]:
logits_result = assert_tensors_equivalent(
    converted_output.logits,
    dense_output.logits,
    name="logits",
    atol=EXPERIMENT["atol"],
    rtol=EXPERIMENT["rtol"],
)
hidden_state_results = []
for index, (actual, expected) in enumerate(
    zip(converted_output.hidden_states, dense_output.hidden_states, strict=True)
):
    hidden_state_results.append(
        assert_tensors_equivalent(
            actual,
            expected,
            name=f"hidden_states[{index}]",
            atol=EXPERIMENT["atol"],
            rtol=EXPERIMENT["rtol"],
        )
    )

for index in selected_layers:
    wrapper = model.model.layers[index].mlp
    assert isinstance(wrapper, RoutedMLP)
    assert wrapper.anchor_mlp is original_mlps[index]
    assert fingerprint_module(wrapper.anchor_mlp) == anchor_fingerprints[index]

assert logits_result.exact
assert all(result.exact for result in hidden_state_results)
validation_summary = {
    "feature_id": EXPERIMENT["feature_id"],
    "converted_layers": conversion_report.converted_layers,
    "exact_logits": logits_result.exact,
    "exact_hidden_states": len(hidden_state_results),
    "anchor_fingerprints_preserved": len(anchor_fingerprints),
    "trainable_parameters_after_conversion": trainable_parameters_after_conversion,
    "all_parameters_frozen": trainable_parameters_after_conversion == 0,
}
print("NB-002 validation passed", validation_summary)